# p53 Triple Mutant MD Simulation (API-based)

**Mutation:** A189S + M133L + S95T

This notebook uses the ESMFold API (no local ML dependencies) for structure prediction,
then runs GROMACS MD simulation.

## Why API-based?
- ColabFold/ESMFold local installs have complex JAX/CUDA dependencies
- ESMFold API at esmatlas.com is free and handles everything server-side
- Only requires `requests` library (pre-installed in Colab)

---
## STEP 1: Setup & Install GROMACS

In [ ]:
#@title Step 1: Install GROMACS and dependencies
import os
import subprocess
import requests
from pathlib import Path

# Create working directory
WORK_DIR = Path("/content/p53_md")
WORK_DIR.mkdir(exist_ok=True)
os.chdir(WORK_DIR)

print("Installing GROMACS...")
!apt-get update -qq
!apt-get install -qq -y gromacs > /dev/null 2>&1

# Verify installation
result = subprocess.run(["gmx", "--version"], capture_output=True, text=True)
if "GROMACS" in result.stdout:
    version_line = [l for l in result.stdout.split('\n') if 'GROMACS version' in l][0]
    print(f"✅ {version_line.strip()}")
else:
    print("❌ GROMACS installation failed")
    raise RuntimeError("GROMACS not installed")

print(f"\n📁 Working directory: {WORK_DIR}")

---
## STEP 2: Predict Mutant Structure via ESMFold API

Uses the free ESMFold API at esmatlas.com - no local ML packages needed.

In [ ]:
#@title Step 2: Predict structure using ESMFold API
import requests
import time

# p53 DNA-binding domain (94-312) with triple mutation: A189S, M133L, S95T
# Original: residue 95=S, 133=M, 189=A
# Mutant:   residue 95=T, 133=L, 189=S

# Wild-type p53 DBD sequence (residues 94-312)
WT_SEQUENCE = (
    "SSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPG"
    "TRVRAMAIYKQSQHMTEVVRRCPHHERCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRH"
    "SVVVPYEPPEVGSDCTTIHYNYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRV"
    "CACPGRDRRTEEENLRKKGEPHHELPPGSTKRALPNNT"
)

# Apply mutations (0-indexed positions relative to sequence start at residue 94)
# S95T: position 1 (95-94=1), S->T
# M133L: position 39 (133-94=39), M->L  
# A189S: position 95 (189-94=95), A->S

def apply_mutations(seq, mutations):
    """Apply mutations to sequence. mutations = [(pos_0indexed, new_aa), ...]"""
    seq_list = list(seq)
    for pos, new_aa in mutations:
        old_aa = seq_list[pos]
        seq_list[pos] = new_aa
        print(f"  Position {pos}: {old_aa} -> {new_aa}")
    return ''.join(seq_list)

print("Applying mutations to p53 DBD sequence:")
mutations = [
    (1, 'T'),   # S95T (position 95 - 94 = 1)
    (39, 'L'),  # M133L (position 133 - 94 = 39)
    (95, 'S'),  # A189S (position 189 - 94 = 95)
]

MUTANT_SEQUENCE = apply_mutations(WT_SEQUENCE, mutations)
print(f"\nSequence length: {len(MUTANT_SEQUENCE)} residues")

# Call ESMFold API
print("\n📡 Calling ESMFold API (this may take 1-2 minutes)...")
ESMFOLD_API = "https://api.esmatlas.com/foldSequence/v1/pdb/"

start_time = time.time()
response = requests.post(
    ESMFOLD_API,
    data=MUTANT_SEQUENCE,
    headers={"Content-Type": "text/plain"},
    timeout=300  # 5 minute timeout
)

if response.status_code == 200:
    pdb_content = response.text
    elapsed = time.time() - start_time
    print(f"✅ Structure predicted in {elapsed:.1f} seconds")
    
    # Save PDB file
    pdb_path = WORK_DIR / "mutant.pdb"
    with open(pdb_path, 'w') as f:
        f.write(pdb_content)
    print(f"📁 Saved to: {pdb_path}")
    
    # Count atoms
    n_atoms = len([l for l in pdb_content.split('\n') if l.startswith('ATOM')])
    print(f"📊 Structure has {n_atoms} atoms")
else:
    print(f"❌ API error: {response.status_code}")
    print(response.text)
    raise RuntimeError("ESMFold API failed")

---
## STEP 3: Prepare System for MD

Generate topology, solvate, add ions, minimize energy.

In [ ]:
#@title Step 3: Prepare system (topology, solvation, minimization)
import subprocess

def run_gmx(cmd, input_text=None):
    """Run GROMACS command with error handling."""
    full_cmd = f"gmx {cmd}"
    result = subprocess.run(
        full_cmd,
        shell=True,
        input=input_text,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print(f"STDERR: {result.stderr[-2000:]}")
        raise RuntimeError(f"Command failed: {full_cmd}")
    return result

os.chdir(WORK_DIR)

# Step 3a: Generate topology with AMBER99SB-ILDN force field
print("📋 Generating topology...")
run_gmx(
    "pdb2gmx -f mutant.pdb -o processed.gro -water tip3p -ignh",
    input_text="6\n"  # AMBER99SB-ILDN
)
print("  ✅ Topology generated")

# Step 3b: Create simulation box (dodecahedron, 1.2nm buffer)
print("📦 Creating simulation box...")
run_gmx("editconf -f processed.gro -o boxed.gro -c -d 1.2 -bt dodecahedron")
print("  ✅ Box created")

# Step 3c: Solvate
print("💧 Adding water...")
run_gmx("solvate -cp boxed.gro -cs spc216.gro -o solvated.gro -p topol.top")

# Count waters
with open("solvated.gro") as f:
    n_sol = sum(1 for line in f if 'SOL' in line) // 3
print(f"  ✅ Added {n_sol} water molecules")

# Step 3d: Create ions.mdp
ions_mdp = """
; Ions
integrator = steep
emtol = 1000.0
nsteps = 50000
"""
with open("ions.mdp", "w") as f:
    f.write(ions_mdp)

# Step 3e: Add ions (neutralize + 0.15M NaCl)
print("🧂 Adding ions...")
run_gmx("grompp -f ions.mdp -c solvated.gro -p topol.top -o ions.tpr -maxwarn 1")
run_gmx("genion -s ions.tpr -o ionized.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15",
        input_text="SOL\n")
print("  ✅ System neutralized")

# Step 3f: Energy minimization
print("⚡ Energy minimization...")
em_mdp = """
; Energy minimization
integrator = steep
emtol = 500.0
emstep = 0.01
nsteps = 50000

nstlist = 10
cutoff-scheme = Verlet
ns_type = grid
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
pbc = xyz
"""
with open("em.mdp", "w") as f:
    f.write(em_mdp)

run_gmx("grompp -f em.mdp -c ionized.gro -p topol.top -o em.tpr")
result = run_gmx("mdrun -v -deffnm em -ntmpi 1")

# Check final energy
with open("em.log") as f:
    for line in f:
        if "Potential Energy" in line:
            print(f"  ✅ {line.strip()}")
            break

print("\n✅ System preparation complete!")

---
## STEP 4: Run MD Simulation

NVT equilibration (100ps) → NPT equilibration (100ps) → Production MD (10ns)

In [ ]:
#@title Step 4: Run MD simulation
import time

os.chdir(WORK_DIR)

# Create index file with protein group
print("📋 Creating index groups...")
run_gmx("make_ndx -f em.gro -o index.ndx", input_text="q\n")

# NVT equilibration MDP (100 ps with position restraints)
nvt_mdp = """
; NVT equilibration
integrator = md
nsteps = 50000      ; 100 ps
dt = 0.002

; Output
nstxout = 5000
nstvout = 5000
nstenergy = 500
nstlog = 500
nstxout-compressed = 1000

; Neighbor searching
cutoff-scheme = Verlet
nstlist = 10
ns_type = grid
pbc = xyz
rlist = 1.0

; Electrostatics
coulombtype = PME
rcoulomb = 1.0
pme_order = 4
fourierspacing = 0.12

; VdW
vdwtype = Cut-off
rvdw = 1.0
DispCorr = EnerPres

; Temperature coupling
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau_t = 0.1 0.1
ref_t = 300 300

; Constraints
constraints = h-bonds
constraint_algorithm = lincs

; Position restraints
define = -DPOSRES
"""
with open("nvt.mdp", "w") as f:
    f.write(nvt_mdp)

# NPT equilibration MDP (100 ps)
npt_mdp = """
; NPT equilibration
integrator = md
nsteps = 50000      ; 100 ps
dt = 0.002

; Output
nstxout = 5000
nstvout = 5000
nstenergy = 500
nstlog = 500
nstxout-compressed = 1000

; Neighbor searching
cutoff-scheme = Verlet
nstlist = 10
ns_type = grid
pbc = xyz
rlist = 1.0

; Electrostatics
coulombtype = PME
rcoulomb = 1.0
pme_order = 4
fourierspacing = 0.12

; VdW
vdwtype = Cut-off
rvdw = 1.0
DispCorr = EnerPres

; Temperature coupling
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau_t = 0.1 0.1
ref_t = 300 300

; Pressure coupling
pcoupl = Parrinello-Rahman
pcoupltype = isotropic
tau_p = 2.0
ref_p = 1.0
compressibility = 4.5e-5

; Constraints
constraints = h-bonds
constraint_algorithm = lincs

; Position restraints
define = -DPOSRES
"""
with open("npt.mdp", "w") as f:
    f.write(npt_mdp)

# Production MD MDP (10 ns)
md_mdp = """
; Production MD
integrator = md
nsteps = 5000000    ; 10 ns
dt = 0.002

; Output
nstxout = 0
nstvout = 0
nstenergy = 5000
nstlog = 5000
nstxout-compressed = 5000  ; Save every 10 ps
compressed-x-grps = Protein

; Neighbor searching
cutoff-scheme = Verlet
nstlist = 10
ns_type = grid
pbc = xyz
rlist = 1.0

; Electrostatics
coulombtype = PME
rcoulomb = 1.0
pme_order = 4
fourierspacing = 0.12

; VdW
vdwtype = Cut-off
rvdw = 1.0
DispCorr = EnerPres

; Temperature coupling
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau_t = 0.1 0.1
ref_t = 300 300

; Pressure coupling
pcoupl = Parrinello-Rahman
pcoupltype = isotropic
tau_p = 2.0
ref_p = 1.0
compressibility = 4.5e-5

; Constraints
constraints = h-bonds
constraint_algorithm = lincs

; No position restraints in production
continuation = yes
gen_vel = no
"""
with open("md.mdp", "w") as f:
    f.write(md_mdp)

# Run NVT equilibration
print("\n🌡️ NVT Equilibration (100 ps)...")
start = time.time()
run_gmx("grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -n index.ndx -o nvt.tpr -maxwarn 1")
run_gmx("mdrun -deffnm nvt -ntmpi 1")
print(f"  ✅ NVT complete ({time.time()-start:.0f}s)")

# Run NPT equilibration
print("\n📊 NPT Equilibration (100 ps)...")
start = time.time()
run_gmx("grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -n index.ndx -o npt.tpr -maxwarn 1")
run_gmx("mdrun -deffnm npt -ntmpi 1")
print(f"  ✅ NPT complete ({time.time()-start:.0f}s)")

# Run production MD
print("\n🚀 Production MD (10 ns)...")
print("   This will take ~30-60 minutes on Colab GPU")
start = time.time()
run_gmx("grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -n index.ndx -o md.tpr")
run_gmx("mdrun -deffnm md -ntmpi 1")
elapsed = time.time() - start
print(f"  ✅ Production MD complete ({elapsed/60:.1f} min)")

print("\n" + "="*50)
print("✅ MD SIMULATION COMPLETE!")
print("="*50)

---
## STEP 5: Analyze Results

Calculate RMSD, RMSF, and check stability.

In [ ]:
#@title Step 5: Analyze trajectory
import subprocess
import numpy as np
import matplotlib.pyplot as plt

os.chdir(WORK_DIR)

print("📊 Analyzing MD trajectory...\n")

# Calculate RMSD (backbone)
print("Calculating RMSD...")
run_gmx("rms -s md.tpr -f md.xtc -o rmsd.xvg -tu ns", input_text="4\n4\n")  # Backbone

# Calculate RMSF (C-alpha)
print("Calculating RMSF...")
run_gmx("rmsf -s md.tpr -f md.xtc -o rmsf.xvg -res", input_text="3\n")  # C-alpha

# Parse RMSD
def parse_xvg(filename):
    x, y = [], []
    with open(filename) as f:
        for line in f:
            if line.startswith(('#', '@')):
                continue
            parts = line.split()
            if len(parts) >= 2:
                x.append(float(parts[0]))
                y.append(float(parts[1]))
    return np.array(x), np.array(y)

time_rmsd, rmsd = parse_xvg("rmsd.xvg")
res_rmsf, rmsf = parse_xvg("rmsf.xvg")

# Plot RMSD
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(time_rmsd, rmsd, 'b-', linewidth=0.8)
axes[0].axhline(y=np.mean(rmsd[-100:]), color='r', linestyle='--', 
                label=f'Final avg: {np.mean(rmsd[-100:]):.2f} nm')
axes[0].set_xlabel('Time (ns)', fontsize=12)
axes[0].set_ylabel('RMSD (nm)', fontsize=12)
axes[0].set_title('Backbone RMSD vs Time', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot RMSF with mutation sites highlighted
mutation_positions = [95, 133, 189]  # Actual residue numbers
# Convert to 0-indexed for our sequence (starts at residue 94)
mutation_indices = [p - 94 for p in mutation_positions]

axes[1].plot(res_rmsf, rmsf, 'b-', linewidth=0.8)
for idx, pos in zip(mutation_indices, mutation_positions):
    if idx < len(rmsf):
        axes[1].axvline(x=idx+1, color='r', linestyle='--', alpha=0.7)
        axes[1].annotate(f'{pos}', (idx+1, rmsf[idx] if idx < len(rmsf) else 0.2), 
                        xytext=(5, 10), textcoords='offset points', fontsize=9, color='r')

axes[1].set_xlabel('Residue', fontsize=12)
axes[1].set_ylabel('RMSF (nm)', fontsize=12)
axes[1].set_title('Per-residue RMSF (red = mutation sites)', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('md_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary statistics
print("\n" + "="*50)
print("ANALYSIS SUMMARY")
print("="*50)
print(f"\nRMSD Statistics:")
print(f"  Initial (first 1 ns):  {np.mean(rmsd[time_rmsd < 1]):.3f} nm")
print(f"  Final (last 2 ns):     {np.mean(rmsd[time_rmsd > 8]):.3f} nm")
print(f"  Average (full):        {np.mean(rmsd):.3f} nm")
print(f"  Maximum:               {np.max(rmsd):.3f} nm")

# Check stability
final_rmsd = np.mean(rmsd[-100:]) * 10  # Convert to Angstroms
if final_rmsd < 2.0:
    stability = "EXCELLENT - very stable structure"
elif final_rmsd < 3.0:
    stability = "GOOD - stable structure"
elif final_rmsd < 4.0:
    stability = "ACCEPTABLE - some flexibility"
else:
    stability = "CONCERNING - significant drift"

print(f"\n📊 Stability Assessment: {stability}")
print(f"   (Final RMSD: {final_rmsd:.2f} Å)")

# RMSF at mutation sites
print(f"\nRMSF at Mutation Sites:")
for idx, pos in zip(mutation_indices, mutation_positions):
    if idx < len(rmsf):
        rmsf_val = rmsf[idx] * 10  # Convert to Angstroms
        print(f"  Position {pos}: {rmsf_val:.2f} Å")

---
## STEP 6: Save Results

In [ ]:
#@title Step 6: Package and download results
import shutil
from datetime import datetime

os.chdir(WORK_DIR)

# Create results directory
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = Path(f"results_{timestamp}")
results_dir.mkdir(exist_ok=True)

# Copy key files
files_to_save = [
    "mutant.pdb",       # Input structure
    "md.tpr",           # Run input
    "md.xtc",           # Trajectory
    "md.gro",           # Final structure
    "md.edr",           # Energy file
    "rmsd.xvg",         # RMSD data
    "rmsf.xvg",         # RMSF data
    "md_analysis.png",  # Analysis plot
    "topol.top",        # Topology
]

print("📦 Packaging results...")
for f in files_to_save:
    if Path(f).exists():
        shutil.copy(f, results_dir / f)
        print(f"  ✓ {f}")

# Create summary file
summary = f"""
p53 Triple Mutant MD Simulation Results
========================================
Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}

Mutation: A189S + M133L + S95T
Sequence: p53 DNA-binding domain (residues 94-312)
Structure prediction: ESMFold API

Simulation Parameters:
- Force field: AMBER99SB-ILDN
- Water model: TIP3P
- Temperature: 300 K
- Pressure: 1 bar
- Production run: 10 ns

Results:
- Final RMSD: {np.mean(rmsd[-100:])*10:.2f} Å
- Average RMSD: {np.mean(rmsd)*10:.2f} Å
- Stability: {stability}

Files:
- md.xtc: Trajectory (10 ns, every 10 ps)
- md.gro: Final structure
- rmsd.xvg, rmsf.xvg: Analysis data
- md_analysis.png: RMSD/RMSF plots
"""

with open(results_dir / "SUMMARY.txt", "w") as f:
    f.write(summary)

# Create zip archive
zip_name = f"p53_md_results_{timestamp}"
shutil.make_archive(zip_name, 'zip', results_dir)
print(f"\n✅ Results saved to: {zip_name}.zip")

# Download link (Colab)
try:
    from google.colab import files
    files.download(f"{zip_name}.zip")
    print("📥 Download started!")
except:
    print(f"📁 Results at: {WORK_DIR}/{zip_name}.zip")